In [1]:
from sts.dio.equity import TickerDatabase as Ticker
import plotly.io as pio
import pandas as pd
import numpy as np
from sts.quant.consolidate import (
    get_consolidate_score_mm_multi_horizon,
)  # get_con_score_mm_multi_horizon
from sts.quant.breakout import get_breakout_mm_multi_horizon
from sts.quant.candle import Candle
from sts.quant.volatility import atr
import os
from datetime import date, datetime
import traceback
import plotly.graph_objs as go
from sts.dio.symbols import sp500_symbol_table
from IPython.display import clear_output

pd.options.plotting.backend = "plotly"
pio.renderers.default = os.environ.get("PLOTLY_RENDERER", "notebook")

In [2]:
resample_rule = "1D"
rule_period_map = {"1D": "3y", "1W": "10y", "1M": "10y"}
period = rule_period_map[resample_rule]
consolidate_score_all_df = {}
syms_with_error = []
for n, row in sp500_symbol_table.iterrows():
    try:
        sym = row["YahooSym"]
        print((n, sym), end=", ")
        ticker = Ticker()
        df = ticker.history(sym, period=period)
        df = df[~df.index.duplicated()]
        df = Candle(df)
        if len(df) > 252:
            con_score_dict = get_consolidate_score_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.1)
            breakout_score_dict = get_breakout_mm_multi_horizon(df.log(), window=20, buffer_threshold=0.1)
            volatilility = atr(df.log(), window=20)
            for key in con_score_dict.keys():
                ts = con_score_dict[key]
                # ts = ts.rolling(window = 3).mean()
                consolidate_score_all_df[(sym, "Con%s" % (key))] = ts
            for key in breakout_score_dict.keys():
                ts = breakout_score_dict[key]
                consolidate_score_all_df[(sym, "Break%s" % (key))] = ts
            consolidate_score_all_df[(sym, "Vol")] = volatilility * np.sqrt(252)
            clear_output()
    except Exception as e:
        syms_with_error.append((sym, str(traceback.print_exc())))
consolidate_score_all_df = pd.DataFrame(consolidate_score_all_df)

ValueError: cannot reindex on an axis with duplicate labels

In [3]:
all_df = None
for key in consolidate_score_all_df.keys():
    ts = consolidate_score_all_df[key]
    print(key)
    ts.name = key
    if ts.index[-1] > datetime(2025, 8, 20):
        ts = ts[~ts.index.duplicated()]
        all_df = pd.concat([all_df, ts], axis=1)

('MMM', 'ConDailyScore')
('MMM', 'ConWeeklyScore')
('MMM', 'ConMonthlyScore')
('MMM', 'BreakDailyScore')
('MMM', 'BreakWeeklyScore')
('MMM', 'BreakMonthlyScore')
('MMM', 'Vol')
('AOS', 'ConDailyScore')
('AOS', 'ConWeeklyScore')
('AOS', 'ConMonthlyScore')
('AOS', 'BreakDailyScore')
('AOS', 'BreakWeeklyScore')
('AOS', 'BreakMonthlyScore')
('AOS', 'Vol')
('ABT', 'ConDailyScore')
('ABT', 'ConWeeklyScore')
('ABT', 'ConMonthlyScore')
('ABT', 'BreakDailyScore')
('ABT', 'BreakWeeklyScore')
('ABT', 'BreakMonthlyScore')
('ABT', 'Vol')
('ABBV', 'ConDailyScore')
('ABBV', 'ConWeeklyScore')
('ABBV', 'ConMonthlyScore')
('ABBV', 'BreakDailyScore')
('ABBV', 'BreakWeeklyScore')
('ABBV', 'BreakMonthlyScore')
('ABBV', 'Vol')
('ACN', 'ConDailyScore')
('ACN', 'ConWeeklyScore')
('ACN', 'ConMonthlyScore')
('ACN', 'BreakDailyScore')
('ACN', 'BreakWeeklyScore')
('ACN', 'BreakMonthlyScore')
('ACN', 'Vol')
('ADM', 'ConDailyScore')
('ADM', 'ConWeeklyScore')
('ADM', 'ConMonthlyScore')
('ADM', 'BreakDailyScore')
('A

In [9]:
consolidate_score_all_df = all_df.ffill()

## Consolidated Score time series

if we check the breakout at weekly level. 
* when it break down, can be view as a down pull back setup
* when it break up start, it can be viewed as a overall market sentiment change. 

In [10]:
for score_key in ["ConMonthlyScore", "ConWeeklyScore", "ConDailyScore"]:
    cons_df = consolidate_score_all_df.loc[:, consolidate_score_all_df.columns.get_level_values(1) == score_key]
    fig = cons_df.mean(axis=1, skipna=True).plot(height=400)
    fig.show()

In [11]:
for score_key in ["BreakMonthlyScore", "BreakWeeklyScore", "BreakDailyScore"]:
    cons_df = consolidate_score_all_df.loc[:, consolidate_score_all_df.columns.get_level_values(1) == score_key]
    fig = cons_df.mean(axis=1, skipna=True).plot(height=400)
    fig.show()